In [0]:
# ==========================================================
# 02_silver_indicador_uf
# Camada: Silver
# Fonte: 2026-11-01_uf.parquet (Bronze)
# Destino: default.indicador_uf
#
# RESPONSABILIDADE
# Transformar a publicação oficial do Indicador Criança
# Alfabetizada (nível UF) em tabela Silver confiável.
#
# NOTA DE ARQUITETURA
# A fonte correta é uf.parquet, não o arquivo de metas por UF.
# O arquivo de metas também possui taxa_alfabetizacao, o que
# mascara o erro de fonte — mesmo problema corrigido no 01.
#
# NOTA SOBRE AS PROPORÇÕES
# As 9 colunas proporcao_aluno_nivel_* existem no arquivo mas
# estão 100% nulas nesta granularidade. São mantidas por
# compatibilidade de schema com a Silver municipal.
#
# NOTA DE PERMISSÃO
# O schema silver pertence a outro usuário. A gravação ocorre
# em default, conforme adaptação documentada na seção 10.
# ==========================================================

from pyspark.sql import functions as F

CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"
SCHEMA_DESTINO = "default"
TABELA_DESTINO = "indicador_uf"

BASE_BRONZE = (
    "/Workspace/Users/rm374983@fiap.com.br/"
    "postech-2iast-fase3-modelo-preditivo-g28/dados/bronze"
)
ARQUIVO_ORIGEM = "2026-11-01_uf.parquet"
CAMINHO_ORIGEM = f"{BASE_BRONZE}/{ARQUIVO_ORIGEM}"

TABELA_FULL = f"{CATALOGO}.{SCHEMA_DESTINO}.{TABELA_DESTINO}"

print(f"Origem : {CAMINHO_ORIGEM}")
print(f"Destino: {TABELA_FULL}")

In [0]:
# ==========================================================
# Célula 2 — Leitura da Bronze
# ==========================================================

df_raw = spark.read.parquet(CAMINHO_ORIGEM)

print(f"Registros lidos: {df_raw.count():,}")
print(f"Colunas: {len(df_raw.columns)}")
df_raw.printSchema()

In [0]:
# ==========================================================
# Célula 3 — Transformação e tipagem
#
# Decisões:
# - serie é mantida: parte da granularidade, mesmo sendo
#   sempre '2' neste arquivo.
# - As 9 proporções são mantidas por compatibilidade de
#   schema, ainda que nulas nesta granularidade.
# - Dedupe por (ano, sigla_uf, serie, rede).
# ==========================================================

df_silver = (
    df_raw
    .select(
        F.col("ano").cast("int").alias("ano"),
        F.col("sigla_uf").cast("string").alias("sigla_uf"),
        F.col("serie").cast("string").alias("serie"),
        F.col("rede").cast("string").alias("rede"),
        F.col("taxa_alfabetizacao").cast("double").alias("taxa_alfabetizacao"),
        F.col("media_portugues").cast("double").alias("media_portugues"),
        *[
            F.col(f"proporcao_aluno_nivel_{n}").cast("double").alias(f"proporcao_aluno_nivel_{n}")
            for n in range(9)
        ],
    )
    .filter(F.col("ano").isNotNull() & F.col("sigla_uf").isNotNull())
    .dropDuplicates(["ano", "sigla_uf", "serie", "rede"])
)

print(f"Registros após transformação: {df_silver.count():,}")

In [0]:
# ==========================================================
# Célula 4 — Rastreabilidade
# ingested_at, source e version em todos os registros.
# ==========================================================

df_silver = (
    df_silver
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source", F.lit(ARQUIVO_ORIGEM))
    .withColumn("version", F.lit("v2"))
)

df_silver.select(
    "ano", "sigla_uf", "rede", "taxa_alfabetizacao",
    "ingested_at", "source", "version"
).show(5, truncate=False)

In [0]:
# ==========================================================
# Célula 5 — Data Quality
# Regras: completude, domínio e unicidade.
# Falhas críticas bloqueiam a gravação.
# ==========================================================

total = df_silver.count()

nulos_taxa = df_silver.filter(F.col("taxa_alfabetizacao").isNull()).count()
nulos_uf = df_silver.filter(F.col("sigla_uf").isNull()).count()

fora_faixa = df_silver.filter(
    (F.col("taxa_alfabetizacao") < 0) | (F.col("taxa_alfabetizacao") > 100)
).count()

# Domínio: 27 UFs brasileiras
UFS_VALIDAS = [
    "AC","AL","AM","AP","BA","CE","DF","ES","GO","MA","MG","MS","MT",
    "PA","PB","PE","PI","PR","RJ","RN","RO","RR","RS","SC","SE","SP","TO"
]
ufs_invalidas = df_silver.filter(~F.col("sigla_uf").isin(UFS_VALIDAS)).count()

chave = ["ano", "sigla_uf", "serie", "rede"]
duplicatas = total - df_silver.dropDuplicates(chave).count()

print("=" * 55)
print("DATA QUALITY — indicador_uf")
print("=" * 55)
print(f"Registros totais     : {total:,}")
print(f"Taxa nula            : {nulos_taxa:,} ({nulos_taxa/total*100:.2f}%)")
print(f"UF nula              : {nulos_uf:,} ({nulos_uf/total*100:.2f}%)")
print(f"Taxa fora de 0-100   : {fora_faixa:,}")
print(f"UF fora do domínio   : {ufs_invalidas:,}")
print(f"Duplicatas na chave  : {duplicatas:,}")
print(f"Anos                 : {[r[0] for r in df_silver.select('ano').distinct().orderBy('ano').collect()]}")
print(f"UFs distintas        : {df_silver.select('sigla_uf').distinct().count():,}")
print(f"Redes                : {[r[0] for r in df_silver.select('rede').distinct().orderBy('rede').collect()]}")
print("=" * 55)

assert nulos_taxa == 0, "FALHA CRÍTICA: taxa_alfabetizacao nula"
assert fora_faixa == 0, "FALHA CRÍTICA: taxa fora do domínio 0-100"
assert ufs_invalidas == 0, "FALHA CRÍTICA: sigla_uf fora do domínio"
assert duplicatas == 0, "FALHA CRÍTICA: duplicatas na chave"
print("Todas as regras críticas passaram.")

In [0]:
# ==========================================================
# Célula 6 — Gravação da Silver
# ==========================================================

(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_FULL))

print(f"Gravado em {TABELA_FULL}")
print(f"Registros: {spark.table(TABELA_FULL).count():,}")

In [0]:
# ==========================================================
# Célula 7 — Validação pós-gravação
# ==========================================================

df_check = spark.table(TABELA_FULL)

print(f"Registros na tabela : {df_check.count():,}")
print(f"Colunas             : {len(df_check.columns)}")
print(f"Anos                : {[r[0] for r in df_check.select('ano').distinct().orderBy('ano').collect()]}")
print(f"UFs                 : {df_check.select('sigla_uf').distinct().count():,}")

print("\nAmostra:")
df_check.select(
    "ano", "sigla_uf", "serie", "rede",
    "taxa_alfabetizacao", "media_portugues"
).show(5, truncate=False)

print("\nDistribuição por rede:")
df_check.groupBy("rede").count().orderBy("rede").show()

In [0]:
UFS_VALIDAS = [
    "AC","AL","AM","AP","BA","CE","DF","ES","GO","MA","MG","MS","MT",
    "PA","PB","PE","PI","PR","RJ","RN","RO","RR","RS","SC","SE","SP","TO"
]

presentes = [r[0] for r in spark.table(
    "dbw_fiap_techchallenge_prod_7405607602465611.default.indicador_uf"
).select("sigla_uf").distinct().collect()]

print("Presentes:", sorted(presentes))
print("Ausentes :", sorted(set(UFS_VALIDAS) - set(presentes)))